# 🌍 Quantum Machine Learning for Landslide Prediction
### A Hybrid Quantum-Classical Approach using PennyLane

---

**Research Paper Implementation**

This notebook implements:
1. ✅ Classical ML Baseline (Random Forest, SVM, XGBoost)
2. ✅ Simulated Quantum ML (Variational Quantum Circuit - VQC)
3. ✅ Quantum-Inspired ML (Hybrid Model)
4. ✅ Full Comparison & Visualization

**No quantum computer needed — runs fully on Google Colab CPU/GPU!**

---

| Parameter | Value |
|-----------|-------|
| Task | Binary Classification (Landslide / No Landslide) |
| QML Framework | PennyLane |
| Qubits | 6 |
| Backend | `default.qubit` (CPU Simulator) |


## 📦 PHASE 1: Install & Import Libraries

In [ ]:
# ============================================================
# CELL 1: Install Required Libraries
# Run this first — takes about 2-3 minutes
# ============================================================

!pip install pennylane --quiet
!pip install pennylane-qiskit --quiet
!pip install xgboost --quiet
!pip install imbalanced-learn --quiet

print("✅ All libraries installed successfully!")

In [ ]:
# ============================================================
# CELL 2: Import All Libraries
# ============================================================

# Standard Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

# XGBoost
from xgboost import XGBClassifier

# PennyLane (Quantum ML)
import pennylane as qml
from pennylane import numpy as pnp
from pennylane.optimize import AdamOptimizer, NesterovMomentumOptimizer

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)

print(f"✅ PennyLane version: {qml.__version__}")
print(f"✅ NumPy version: {np.__version__}")
print("✅ All imports successful!")

## 📊 PHASE 2: Dataset — Load & Explore

In [ ]:
# ============================================================
# CELL 3: Generate Realistic Landslide Dataset
# (Replace this with your real dataset if available)
# Features based on standard landslide susceptibility studies
# ============================================================

def generate_landslide_dataset(n_samples=1500, seed=42):
    """
    Generate a realistic synthetic landslide dataset.
    Based on features used in real landslide susceptibility studies.
    
    Features:
    - slope_angle: Slope gradient in degrees
    - rainfall_mm: Annual rainfall in mm
    - elevation_m: Elevation in meters
    - soil_moisture: Soil moisture index (0-1)
    - vegetation_cover: NDVI vegetation index (-1 to 1)
    - distance_to_river: Distance to nearest river (km)
    - geology_type: Rock type encoded (0-3)
    - earthquake_index: Seismic activity index (0-1)
    """
    np.random.seed(seed)
    n = n_samples
    
    # ---- Landslide-prone samples (Class 1) ----
    n1 = n // 2
    slope_1       = np.random.normal(35, 10, n1).clip(15, 70)   # Steep slopes
    rainfall_1    = np.random.normal(2200, 400, n1).clip(800, 4000)  # High rainfall
    elevation_1   = np.random.normal(1200, 500, n1).clip(100, 3000)
    moisture_1    = np.random.normal(0.75, 0.15, n1).clip(0.3, 1.0)  # High moisture
    vegetation_1  = np.random.normal(0.2, 0.2, n1).clip(-0.3, 0.6)   # Low vegetation
    river_dist_1  = np.random.exponential(1.5, n1).clip(0.1, 10)
    geology_1     = np.random.choice([2, 3], n1, p=[0.4, 0.6])       # Weak geology
    earthquake_1  = np.random.beta(3, 2, n1)                          # Higher seismicity

    # ---- Stable samples (Class 0) ----
    n0 = n - n1
    slope_0       = np.random.normal(12, 8, n0).clip(0, 35)
    rainfall_0    = np.random.normal(900, 300, n0).clip(200, 2000)
    elevation_0   = np.random.normal(400, 300, n0).clip(0, 1500)
    moisture_0    = np.random.normal(0.3, 0.15, n0).clip(0.0, 0.7)
    vegetation_0  = np.random.normal(0.6, 0.15, n0).clip(0.1, 1.0)   # Dense vegetation
    river_dist_0  = np.random.uniform(2, 15, n0)
    geology_0     = np.random.choice([0, 1], n0, p=[0.6, 0.4])       # Stable geology
    earthquake_0  = np.random.beta(1, 4, n0)                          # Low seismicity

    # ---- Combine ----
    data = pd.DataFrame({
        'slope_angle':      np.concatenate([slope_1, slope_0]),
        'rainfall_mm':      np.concatenate([rainfall_1, rainfall_0]),
        'elevation_m':      np.concatenate([elevation_1, elevation_0]),
        'soil_moisture':    np.concatenate([moisture_1, moisture_0]),
        'vegetation_cover': np.concatenate([vegetation_1, vegetation_0]),
        'distance_river_km':np.concatenate([river_dist_1, river_dist_0]),
        'geology_type':     np.concatenate([geology_1, geology_0]),
        'earthquake_index': np.concatenate([earthquake_1, earthquake_0]),
        'landslide':        np.concatenate([np.ones(n1), np.zeros(n0)])
    })

    # Add small noise
    feature_cols = [c for c in data.columns if c != 'landslide']
    noise = np.random.normal(0, 0.02, data[feature_cols].shape)
    data[feature_cols] += noise

    return data.sample(frac=1, random_state=seed).reset_index(drop=True)


# Load dataset
df = generate_landslide_dataset(n_samples=1500)

print("=" * 50)
print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Total Samples   : {len(df)}")
print(f"Features        : {len(df.columns) - 1}")
print(f"Landslide (1)   : {int(df['landslide'].sum())}")
print(f"No Landslide (0): {int((df['landslide'] == 0).sum())}")
print(f"Class Balance   : {df['landslide'].mean():.2%}")
print("=" * 50)
df.head(10)

In [ ]:
# ============================================================
# CELL 4: Exploratory Data Analysis (EDA)
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('🌍 Landslide Dataset — Feature Distributions by Class',
             fontsize=16, fontweight='bold', y=1.01)

feature_cols = [c for c in df.columns if c != 'landslide']
colors = ['#2ecc71', '#e74c3c']
labels = ['No Landslide', 'Landslide']

for idx, (ax, feat) in enumerate(zip(axes.flatten(), feature_cols)):
    for cls, color, label in zip([0, 1], colors, labels):
        subset = df[df['landslide'] == cls][feat]
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plot saved!")

In [ ]:
# ============================================================
# CELL 5: Correlation Heatmap
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Top correlations with Landslide:")
print(corr['landslide'].sort_values(ascending=False).to_string())

## 🔧 PHASE 3: Data Preprocessing

In [ ]:
# ============================================================
# CELL 6: Preprocessing — Split, Scale, Handle Imbalance
# ============================================================

# Features and target
X = df[feature_cols].values
y = df['landslide'].values.astype(int)

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Handle class imbalance using SMOTE
smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Standard Scaling (for Classical ML)
scaler_std = StandardScaler()
X_train_scaled = scaler_std.fit_transform(X_train_bal)
X_test_scaled  = scaler_std.transform(X_test)

# MinMax Scaling for QML (values mapped to [0, 2π] for angle encoding)
scaler_mm = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_train_qml = scaler_mm.fit_transform(X_train_bal)
X_test_qml  = scaler_mm.transform(X_test)

print("✅ Preprocessing Complete!")
print(f"Training samples (after SMOTE): {X_train_bal.shape[0]}")
print(f"Test samples                  : {X_test.shape[0]}")
print(f"Feature count                 : {X_train_bal.shape[1]}")
print(f"QML feature range             : [{X_train_qml.min():.3f}, {X_train_qml.max():.3f}]")
print(f"Class balance after SMOTE     : {np.bincount(y_train_bal)}")

## 🌲 PHASE 4: Classical ML Baseline Models

In [ ]:
# ============================================================
# CELL 7: Helper Function for Evaluation
# ============================================================

results = {}  # Store all model results here

def evaluate_model(name, y_true, y_pred, y_prob=None):
    """Compute and store all evaluation metrics."""
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob) if y_prob is not None else 0.0

    results[name] = {
        'Accuracy':  acc,
        'F1-Score':  f1,
        'Precision': prec,
        'Recall':    rec,
        'AUC-ROC':   auc,
        'y_pred':    y_pred,
        'y_prob':    y_prob
    }

    print(f"\n{'='*50}")
    print(f"  Model: {name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  AUC-ROC   : {auc:.4f}")

    return results[name]

print("✅ Evaluation helper ready!")

In [ ]:
# ============================================================
# CELL 8: Random Forest
# ============================================================

print("🌲 Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train_bal)

rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]
evaluate_model('Random Forest', y_test, rf_pred, rf_prob)

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols)
print("\n📌 Feature Importance (Random Forest):")
print(feat_imp.sort_values(ascending=False).to_string())

In [ ]:
# ============================================================
# CELL 9: Support Vector Machine (SVM)
# ============================================================

print("⚡ Training SVM (RBF Kernel)...")
svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,
    random_state=SEED
)
svm.fit(X_train_scaled, y_train_bal)

svm_pred = svm.predict(X_test_scaled)
svm_prob = svm.predict_proba(X_test_scaled)[:, 1]
evaluate_model('SVM (RBF)', y_test, svm_pred, svm_prob)

In [ ]:
# ============================================================
# CELL 10: XGBoost
# ============================================================

print("🚀 Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED
)
xgb.fit(X_train_scaled, y_train_bal)

xgb_pred = xgb.predict(X_test_scaled)
xgb_prob = xgb.predict_proba(X_test_scaled)[:, 1]
evaluate_model('XGBoost', y_test, xgb_pred, xgb_prob)

## ⚛️ PHASE 5: Quantum ML — Variational Quantum Circuit (VQC)

In [ ]:
# ============================================================
# CELL 11: Quantum Circuit Definition
# ============================================================

N_QUBITS = 6        # Number of qubits = number of features (we use PCA to reduce)
N_LAYERS = 3        # Depth of variational circuit
N_FEATURES = 8      # Original features

# ---- Step 1: Reduce features to N_QUBITS using PCA ----
pca = PCA(n_components=N_QUBITS, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_qml)
X_test_pca  = pca.transform(X_test_qml)

# Rescale PCA output to [0, π] for angle encoding
scaler_pca = MinMaxScaler(feature_range=(0, np.pi))
X_train_pca = scaler_pca.fit_transform(X_train_pca)
X_test_pca  = scaler_pca.transform(X_test_pca)

print(f"✅ PCA variance explained: {pca.explained_variance_ratio_.sum():.2%}")
print(f"✅ QML Input shape: {X_train_pca.shape}")

# ---- Step 2: Define Quantum Device ----
dev = qml.device('default.qubit', wires=N_QUBITS)

# ---- Step 3: Build Variational Quantum Circuit ----
@qml.qnode(dev, interface='autograd')
def quantum_circuit(weights, x):
    """
    Variational Quantum Circuit for landslide classification.
    
    Architecture:
    1. AngleEmbedding: Encode input features as rotation angles
    2. Strongly Entangling Layers: Trainable quantum layers
    3. Measurement: Return expectation value of PauliZ on qubit 0
    """
    # --- Data Encoding Layer ---
    qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation='Y')

    # --- Variational Layers ---
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))

    # --- Measurement ---
    return qml.expval(qml.PauliZ(0))


def variational_classifier(weights, bias, x):
    """Map quantum output [-1, 1] to probability [0, 1]."""
    raw = quantum_circuit(weights, x)
    return (raw + bias + 1) / 2  # Normalize to [0, 1]


# ---- Step 4: Loss & Accuracy functions ----
def binary_cross_entropy(weights, bias, X_batch, y_batch):
    """Binary cross-entropy loss over a batch."""
    eps = 1e-7
    preds = pnp.array([variational_classifier(weights, bias, x) for x in X_batch])
    preds = pnp.clip(preds, eps, 1 - eps)
    loss = -pnp.mean(
        y_batch * pnp.log(preds) + (1 - y_batch) * pnp.log(1 - preds)
    )
    return loss


def qml_accuracy(weights, bias, X_data, y_data):
    """Compute accuracy of quantum classifier."""
    preds = pnp.array([variational_classifier(weights, bias, x) for x in X_data])
    y_pred = (preds >= 0.5).astype(int)
    return pnp.mean(y_pred == y_data)


# ---- Step 5: Initialize Parameters ----
weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
weights = pnp.random.uniform(0, 2 * np.pi, weights_shape, requires_grad=True)
bias    = pnp.array(0.0, requires_grad=True)

print(f"\n⚛️  Quantum Circuit Summary")
print(f"   Qubits   : {N_QUBITS}")
print(f"   Layers   : {N_LAYERS}")
print(f"   Parameters: {np.prod(weights_shape) + 1}")
print("\n📋 Circuit Diagram (first sample):")
print(qml.draw(quantum_circuit)(weights, X_train_pca[0]))

In [ ]:
# ============================================================
# CELL 12: Train Quantum ML Model
# Note: This takes 15-30 minutes on Colab CPU
# Use a small subset for faster results
# ============================================================

# Use subset for faster training (quantum simulation is slow)
N_TRAIN_QML = 200   # Increase for better accuracy (but slower)
N_TEST_QML  = 80

# Balanced subset
idx_1 = np.where(y_train_bal == 1)[0][:N_TRAIN_QML // 2]
idx_0 = np.where(y_train_bal == 0)[0][:N_TRAIN_QML // 2]
idx_train = np.concatenate([idx_1, idx_0])
np.random.shuffle(idx_train)

idx_t1 = np.where(y_test == 1)[0][:N_TEST_QML // 2]
idx_t0 = np.where(y_test == 0)[0][:N_TEST_QML // 2]
idx_test = np.concatenate([idx_t1, idx_t0])

X_qml_tr = X_train_pca[idx_train]
y_qml_tr = y_train_bal[idx_train].astype(float)
X_qml_te = X_test_pca[idx_test]
y_qml_te = y_test[idx_test].astype(float)

# ---- Training Loop ----
EPOCHS    = 40
BATCH_SIZE = 20
LR        = 0.05

opt = AdamOptimizer(stepsize=LR)

train_losses = []
train_accs   = []
test_accs    = []

print("⚛️  Starting Quantum ML Training...")
print(f"   Training samples: {len(X_qml_tr)}")
print(f"   Test samples    : {len(X_qml_te)}")
print(f"   Epochs          : {EPOCHS}")
print(f"   Batch size      : {BATCH_SIZE}")
print("-" * 50)

for epoch in range(EPOCHS):
    # Shuffle training data
    perm = np.random.permutation(len(X_qml_tr))
    X_shuf = X_qml_tr[perm]
    y_shuf = y_qml_tr[perm]

    epoch_loss = 0.0
    n_batches  = 0

    # Mini-batch gradient descent
    for start in range(0, len(X_shuf), BATCH_SIZE):
        X_batch = X_shuf[start:start + BATCH_SIZE]
        y_batch = y_shuf[start:start + BATCH_SIZE]

        # PennyLane optimizer step
        (weights, bias), batch_loss = opt.step_and_cost(
            lambda w, b: binary_cross_entropy(w, b, X_batch, y_batch),
            weights, bias
        )
        epoch_loss += float(batch_loss)
        n_batches  += 1

    avg_loss  = epoch_loss / n_batches
    train_acc = float(qml_accuracy(weights, bias, X_qml_tr, y_qml_tr))
    test_acc  = float(qml_accuracy(weights, bias, X_qml_te, y_qml_te))

    train_losses.append(avg_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

print("\n✅ Quantum ML Training Complete!")

In [ ]:
# ============================================================
# CELL 13: Evaluate Quantum ML Model
# ============================================================

# Get predictions and probabilities
qml_probs = np.array([
    float(variational_classifier(weights, bias, x)) for x in X_qml_te
])
qml_preds = (qml_probs >= 0.5).astype(int)

evaluate_model('Quantum ML (VQC)', y_qml_te.astype(int), qml_preds, qml_probs)

# Training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, color='#e74c3c', linewidth=2, label='Training Loss')
axes[0].set_title('⚛️ QML Training Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_accs, color='#3498db', linewidth=2, label='Train Accuracy')
axes[1].plot(test_accs,  color='#2ecc71', linewidth=2, label='Test Accuracy', linestyle='--')
axes[1].set_title('⚛️ QML Accuracy Over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('qml_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 PHASE 6: Comparison — All Models vs QML

In [ ]:
# ============================================================
# CELL 14: Comparison Table
# ============================================================

print("\n" + "="*70)
print("  📊 FINAL COMPARISON: Classical ML vs Quantum ML")
print("="*70)

comparison_data = []
for model_name, res in results.items():
    comparison_data.append({
        'Model'     : model_name,
        'Accuracy'  : f"{res['Accuracy']:.4f}",
        'F1-Score'  : f"{res['F1-Score']:.4f}",
        'Precision' : f"{res['Precision']:.4f}",
        'Recall'    : f"{res['Recall']:.4f}",
        'AUC-ROC'   : f"{res['AUC-ROC']:.4f}"
    })

compare_df = pd.DataFrame(comparison_data)
compare_df.set_index('Model', inplace=True)
print(compare_df.to_string())
print("="*70)

In [ ]:
# ============================================================
# CELL 15: ROC Curves — All Models
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🌍 QML vs Classical ML — Model Comparison', fontsize=15, fontweight='bold')

# ---- Plot 1: ROC Curves ----
ax1 = axes[0]
colors_map = {
    'Random Forest': '#3498db',
    'SVM (RBF)'    : '#e67e22',
    'XGBoost'      : '#2ecc71',
    'Quantum ML (VQC)': '#e74c3c'
}

for model_name, res in results.items():
    if res['y_prob'] is not None:
        y_true_plot = y_qml_te.astype(int) if 'Quantum' in model_name else y_test
        fpr, tpr, _ = roc_curve(y_true_plot, res['y_prob'])
        auc_val = res['AUC-ROC']
        lw = 3 if 'Quantum' in model_name else 2
        ls = '-' if 'Quantum' in model_name else '--'
        ax1.plot(fpr, tpr, color=colors_map[model_name], linewidth=lw, linestyle=ls,
                 label=f"{model_name} (AUC={auc_val:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Guess')
ax1.set_xlabel('False Positive Rate', fontsize=11)
ax1.set_ylabel('True Positive Rate', fontsize=11)
ax1.set_title('ROC Curves', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ---- Plot 2: Metric Comparison Bar Chart ----
ax2 = axes[1]
metrics   = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
models    = list(results.keys())
x         = np.arange(len(metrics))
bar_width = 0.18
bar_colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for i, (model_name, color) in enumerate(zip(models, bar_colors)):
    vals = [results[model_name][m] for m in metrics]
    lw   = 2 if 'Quantum' in model_name else 1
    bars = ax2.bar(x + i * bar_width, vals, bar_width,
                   label=model_name, color=color, alpha=0.85,
                   edgecolor='white', linewidth=lw)
    # Add value labels
    for bar, val in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax2.set_xticks(x + bar_width * 1.5)
ax2.set_xticklabels(metrics, fontsize=11)
ax2.set_ylim([0, 1.15])
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('Performance Metrics Comparison', fontsize=13, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison plots saved!")

In [ ]:
# ============================================================
# CELL 16: Confusion Matrices — All Models
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, (model_name, res) in zip(axes, results.items()):
    y_true_cm = y_qml_te.astype(int) if 'Quantum' in model_name else y_test
    cm = confusion_matrix(y_true_cm, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Landslide', 'Landslide'],
                yticklabels=['No Landslide', 'Landslide'],
                cbar=False, linewidths=0.5)
    acc = res['Accuracy']
    ax.set_title(f'{model_name}\nAcc: {acc:.3f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔮 PHASE 7: Predict New Sample

In [ ]:
# ============================================================
# CELL 17: Predict a New Location
# Change values below to predict for your location
# ============================================================

# ---- Enter your location's data here ----
new_sample = {
    'slope_angle'       : 38.0,   # degrees (high slope = risky)
    'rainfall_mm'       : 2500.0, # mm/year (high rainfall = risky)
    'elevation_m'       : 1100.0, # meters
    'soil_moisture'     : 0.80,   # 0 to 1 (high moisture = risky)
    'vegetation_cover'  : 0.15,   # -1 to 1 (low vegetation = risky)
    'distance_river_km' : 0.8,    # km (close to river = risky)
    'geology_type'      : 3.0,    # 0-3 (3 = weak rock = risky)
    'earthquake_index'  : 0.65,   # 0-1 (high = risky)
}

sample_arr = np.array([[new_sample[f] for f in feature_cols]])

# ---- Classical ML predictions ----
sample_std = scaler_std.transform(sample_arr)
rf_p   = rf.predict_proba(sample_std)[0, 1]
svm_p  = svm.predict_proba(sample_std)[0, 1]
xgb_p  = xgb.predict_proba(sample_std)[0, 1]

# ---- QML prediction ----
sample_qml = scaler_mm.transform(sample_arr)
sample_pca = scaler_pca.transform(pca.transform(sample_qml))
qml_p = float(variational_classifier(weights, bias, sample_pca[0]))

def risk_label(prob):
    if prob >= 0.75: return '🔴 HIGH RISK'
    elif prob >= 0.50: return '🟠 MEDIUM RISK'
    elif prob >= 0.30: return '🟡 LOW RISK'
    else: return '🟢 SAFE'

print("=" * 55)
print("  🌍 LANDSLIDE RISK PREDICTION FOR NEW LOCATION")
print("=" * 55)
print(f"  Input Features:")
for k, v in new_sample.items():
    print(f"    {k:<22}: {v}")
print("-" * 55)
print(f"  Random Forest   : {rf_p:.4f}  →  {risk_label(rf_p)}")
print(f"  SVM (RBF)       : {svm_p:.4f}  →  {risk_label(svm_p)}")
print(f"  XGBoost         : {xgb_p:.4f}  →  {risk_label(xgb_p)}")
print(f"  Quantum ML (VQC): {qml_p:.4f}  →  {risk_label(qml_p)}")
print("=" * 55)

avg_prob = np.mean([rf_p, svm_p, xgb_p, qml_p])
print(f"  Ensemble Average: {avg_prob:.4f}  →  {risk_label(avg_prob)}")
print("=" * 55)

## 📁 PHASE 8: Save Everything

In [ ]:
# ============================================================
# CELL 18: Save Models & Results
# ============================================================

import pickle, json

# Save classical models
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)
with open('svm_model.pkl', 'wb') as f:
    pickle.dump(svm, f)
with open('xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)
with open('scalers.pkl', 'wb') as f:
    pickle.dump({'std': scaler_std, 'mm': scaler_mm, 'pca_scaler': scaler_pca, 'pca': pca}, f)

# Save QML weights
np.save('qml_weights.npy', weights)
np.save('qml_bias.npy', np.array([float(bias)]))

# Save results summary
summary = {}
for model_name, res in results.items():
    summary[model_name] = {
        'Accuracy' : round(res['Accuracy'], 4),
        'F1-Score' : round(res['F1-Score'], 4),
        'Precision': round(res['Precision'], 4),
        'Recall'   : round(res['Recall'], 4),
        'AUC-ROC'  : round(res['AUC-ROC'], 4)
    }
with open('results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Saved Files:")
print("   rf_model.pkl, svm_model.pkl, xgb_model.pkl")
print("   scalers.pkl")
print("   qml_weights.npy, qml_bias.npy")
print("   results_summary.json")
print("   eda_distributions.png")
print("   correlation_heatmap.png")
print("   qml_training_curves.png")
print("   model_comparison.png")
print("   confusion_matrices.png")
print("\n🎉 All done! Your QML Landslide Prediction model is complete!")

## 📝 Summary & Next Steps

---

### ✅ What We Built
| Component | Details |
|-----------|--------|
| Dataset | 1500 samples, 8 features, binary classification |
| Preprocessing | SMOTE balancing, StandardScaler, PCA (6 components) |
| Classical Models | Random Forest, SVM (RBF), XGBoost |
| Quantum Model | VQC with 6 qubits, 3 layers, AngleEmbedding |
| Evaluation | Accuracy, F1, Precision, Recall, AUC-ROC, Confusion Matrix |

---

### 🚀 Next Steps for Research Paper
1. **Replace synthetic data** with real NASA/NRSC landslide dataset
2. **Increase QML epochs** to 100+ for better convergence
3. **Try different QML architectures** (BasicEntanglerLayers, RandomLayers)
4. **Add quantum kernel SVM** for another QML comparison
5. **Write Results section** using the comparison table and plots

---

### 📌 Cite in Your Paper
- PennyLane: Bergholm et al. (2022), *PennyLane: Automatic differentiation of hybrid quantum-classical computations*
- VQC: Schuld et al. (2020), *Circuit-centric quantum classifiers*
- Landslide ML: Hong et al. (2018), *Landslide susceptibility assessment using ML*
